# 뮤직비디오 청각(오디오) 특징 추출

K-pop 뮤직비디오의 오디오 트랙에서 곡의 특징을 나타내는 지표를 추출하는 코드입니다. 유튜브 영상에서 오디오만 다운로드한 뒤, `librosa`와 `webrtcvad`로 다음 특징을 계산합니다.

- **energy / energy_variance** — 음량(RMS) 평균과 변화량
- **speechiness / speech_duration_sec** — 보컬(음성) 구간 비율과 길이
- **spectral_flatness / harmonic_ratio / acoustic_score** — 음향적 특성(어쿠스틱함 정도)

**구성**
1. 환경 설정 — 필요한 패키지 설치
2. 메인 청각 특징 추출 — 전체 곡 목록을 대상으로 오디오 다운로드 및 특징 추출
3. 실패 항목 재추출 — 다운로드/추출에 실패한 곡만 골라내 다시 시도

**참고사항**
- 이 코드는 Google Colab 환경(`/content/...` 경로, `google.colab.files`)을 기준으로 작성되었습니다.
- 원본 곡 목록(`final_1000_ordered(새롭게).csv`)과 추출 결과 파일은 저작권이 있는 곡 정보를 포함하고 있어 이 저장소에는 포함하지 않았습니다.


## 1. 환경 설정

오디오 다운로드/분석에 필요한 패키지(`yt-dlp`, `librosa`, `webrtcvad`, `soundfile`)와 시스템 도구(`ffmpeg`)를 설치합니다.


In [ ]:
!pip install pandas yt-dlp librosa webrtcvad soundfile numpy

In [ ]:
!apt-get install -y ffmpeg nodejs
!pip install -U yt-dlp librosa webrtcvad soundfile

## 2. 메인 청각 특징 추출

`AudioFeatureExtractor` 클래스로 곡 목록(`final_1000_ordered(새롭게).csv`)의 각 유튜브 URL에서 오디오를 다운로드하고, `energy`, `speechiness`, `acoustic_score` 등 오디오 특징을 계산해 `audio_features_output.csv`로 저장합니다.


In [ ]:
import pandas as pd
import yt_dlp
import librosa
import numpy as np
import webrtcvad
import wave
import os
from pathlib import Path

class AudioFeatureExtractor:
    def __init__(self):
        self.vad = webrtcvad.Vad(3)

    def download_audio(self, youtube_url, temp_dir, video_id):
        """유튜브에서 오디오 다운로드 후 실제 저장된 경로 반환"""
        output_template = os.path.join(temp_dir, f"{video_id}.%(ext)s")

        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'wav',
                'preferredquality': '192',
            }],
            'outtmpl': output_template,
            'quiet': True,
            'no_warnings': True,
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(youtube_url, download=True)
            # yt_dlp가 최종적으로 생성한 파일 경로 파악
            actual_filename = ydl.prepare_filename(info).replace(info['ext'], 'wav')
            return actual_filename

    def extract_energy(self, audio_path):
        y, sr = librosa.load(audio_path, sr=None)
        rms = librosa.feature.rms(y=y)[0]
        return {
            'energy': float(np.mean(rms)),
            'energy_variance': float(np.std(rms))
        }

    def extract_speechiness(self, audio_path):
        # VAD를 위해 16kHz로 로드
        y, sr = librosa.load(audio_path, sr=16000, mono=True)

        # 정규화 및 16비트 변환
        y_int = (y * 32767).astype(np.int16)

        frame_duration = 30 # ms
        frame_size = int(16000 * frame_duration / 1000)

        speech_frames = 0
        total_frames = 0

        for i in range(0, len(y_int) - frame_size, frame_size):
            frame = y_int[i:i + frame_size].tobytes()
            total_frames += 1
            if self.vad.is_speech(frame, 16000):
                speech_frames += 1

        speechiness = speech_frames / total_frames if total_frames > 0 else 0
        return {
            'speechiness': speechiness,
            'speech_duration_sec': speechiness * (len(y) / 16000)
        }

    def extract_acousticness(self, audio_path):
        y, sr = librosa.load(audio_path, sr=None)
        flatness = librosa.feature.spectral_flatness(y=y)
        y_harmonic, _ = librosa.effects.hpss(y)
        harmonic_ratio = float(np.sum(y_harmonic**2) / (np.sum(y**2) + 1e-6))

        return {
            'spectral_flatness': float(np.mean(flatness)),
            'harmonic_ratio': harmonic_ratio,
            'acoustic_score': harmonic_ratio * (1 - float(np.mean(flatness)))
        }

    def process_csv(self, input_path, output_path, temp_dir='./temp_audio'):
        Path(temp_dir).mkdir(parents=True, exist_ok=True)
        df = pd.read_csv(input_path)
        results = []

        for idx, row in df.iterrows():
            print(f"[{idx+1}/{len(df)}] Processing: {row['songName']}")
            temp_audio = None
            try:
                # 1. 다운로드 (실제 저장된 경로를 가져옴)
                temp_audio = self.download_audio(row['url'], temp_dir, row['video_id'])

                if not os.path.exists(temp_audio):
                    raise FileNotFoundError(f"파일이 생성되지 않음: {temp_audio}")

                # 2. 특성 추출
                energy = self.extract_energy(temp_audio)
                speech = self.extract_speechiness(temp_audio)
                acoustic = self.extract_acousticness(temp_audio)

                # 3. 결과 정리
                res = {**row.to_dict(), **energy, **speech, **acoustic}
                results.append(res)

            except Exception as e:
                print(f"   ❌ Error: {e}")
            finally:
                # 파일 삭제
                if temp_audio and os.path.exists(temp_audio):
                    os.remove(temp_audio)

        result_df = pd.DataFrame(results)
        result_df.to_csv(output_path, index=False)
        print(f"\n✅ 완료! 파일 저장됨: {output_path}")
        return result_df

# 실행
extractor = AudioFeatureExtractor()
# 본인의 파일명에 맞게 수정하세요
results = extractor.process_csv("/content/final_1000_ordered(새롭게).csv", "audio_features_output.csv")

## 3. 실패 항목 재추출

원본 곡 목록과 추출 결과를 `video_id` 기준으로 병합해 오디오 특징이 비어있는(다운로드/추출 실패) 행만 골라냅니다. 그 실패 목록만 다시 `AudioFeatureExtractor`에 넣어 재다운로드·재추출합니다.


In [ ]:
import pandas as pd

# ===== 원본 결과와 병합해서 청각 정보가 비어있는(추출 실패) 행만 골라내기 =====
original_df = pd.read_csv("final_1000_ordered(새롭게).csv")
success_df = pd.read_csv("audio_features_output.csv")

print("원본 개수:", len(original_df))
print("성공 개수:", len(success_df))

audio_cols = [
    'video_id',
    'energy',
    'energy_variance',
    'speechiness',
    'speech_duration_sec',
    'spectral_flatness',
    'harmonic_ratio',
    'acoustic_score'
]
success_df = success_df[audio_cols]

merged_df = original_df.merge(success_df, on='video_id', how='left')
print("병합 후 개수:", len(merged_df))

failed_df = merged_df[merged_df[audio_cols].isnull().any(axis=1)]
output_path = "/content/청각추출_실패_18개.csv"
failed_df.to_csv(output_path, index=False)
print(f"추출 실패 {len(failed_df)}개 저장: {output_path}")

# ===== 실패한 행만 다시 다운로드해서 재추출 =====
import yt_dlp
import librosa
import numpy as np
import webrtcvad
import os
from pathlib import Path
from google.colab import files

class AudioFeatureExtractor:
    def __init__(self):
        self.vad = webrtcvad.Vad(3)

    def download_audio(self, youtube_url, temp_dir, video_id):
        output_template = os.path.join(temp_dir, f"{video_id}.%(ext)s")
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav', 'preferredquality': '192'}],
            'outtmpl': output_template,
            'quiet': True,
            'no_warnings': True,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(youtube_url, download=True)
            actual_filename = ydl.prepare_filename(info).replace(info['ext'], 'wav')
            return actual_filename

    def extract_energy(self, audio_path):
        y, sr = librosa.load(audio_path, sr=None)
        rms = librosa.feature.rms(y=y)[0]
        return {'energy': float(np.mean(rms)), 'energy_variance': float(np.std(rms))}

    def extract_speechiness(self, audio_path):
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        y_int = (y * 32767).astype(np.int16)
        frame_size = int(16000 * 30 / 1000)
        speech_frames, total_frames = 0, 0
        for i in range(0, len(y_int) - frame_size, frame_size):
            frame = y_int[i:i + frame_size].tobytes()
            total_frames += 1
            if self.vad.is_speech(frame, 16000):
                speech_frames += 1
        speechiness = speech_frames / total_frames if total_frames > 0 else 0
        return {'speechiness': speechiness, 'speech_duration_sec': speechiness * (len(y)/16000)}

    def extract_acousticness(self, audio_path):
        y, sr = librosa.load(audio_path, sr=None)
        flatness = librosa.feature.spectral_flatness(y=y)
        y_harmonic, _ = librosa.effects.hpss(y)
        harmonic_ratio = float(np.sum(y_harmonic**2) / (np.sum(y**2)+1e-6))
        acoustic_score = harmonic_ratio * (1 - float(np.mean(flatness)))
        return {
            'spectral_flatness': float(np.mean(flatness)),
            'harmonic_ratio': harmonic_ratio,
            'acoustic_score': acoustic_score
        }

    def process_csv(self, input_path, output_path, temp_dir='./temp_audio'):
        Path(temp_dir).mkdir(parents=True, exist_ok=True)
        df = pd.read_csv(input_path)
        results = []

        for idx, row in df.iterrows():
            print(f"[{idx+1}/{len(df)}] Processing: {row['songName']}")
            temp_audio = None
            try:
                temp_audio = self.download_audio(row['url'], temp_dir, row['video_id'])
                if not os.path.exists(temp_audio):
                    raise FileNotFoundError(f"파일이 생성되지 않음: {temp_audio}")

                energy = self.extract_energy(temp_audio)
                speech = self.extract_speechiness(temp_audio)
                acoustic = self.extract_acousticness(temp_audio)

                res = {**row.to_dict(), **energy, **speech, **acoustic}
                results.append(res)
            except Exception as e:
                print(f"   ❌ Error: {e}")
            finally:
                if temp_audio and os.path.exists(temp_audio):
                    os.remove(temp_audio)

        result_df = pd.DataFrame(results)
        result_df.to_csv(output_path, index=False)
        print(f"\n✅ 완료! 파일 저장됨: {output_path}")

        # Colab에서 내 PC로 다운로드
        files.download(output_path)
        return result_df

# ====== 실행 ======
extractor = AudioFeatureExtractor()
results = extractor.process_csv(
    input_path="/content/청각추출_실패_18개.csv",
    output_path="/content/청각추출_실패_18개_오디오다시추출완료.csv"
)
